In [2]:
pip install bstabdiff

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 165.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 167.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 150.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 139.2 MB/s  0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [bstabdiff]


In [4]:
pip show bstabdiff

Name: bstabdiff
Version: 0.2.1
Summary: BSTabDiff: Block-Subunit Diffusion Priors for High-Dimensional Tabular Data Generation (NeurIPS 2026)
Home-page: https://www.zadidhabib.com/bstabdiff.html
Author: Al Zadid Sultan Bin Habib, Md Younus Ahamed, Prashnna Kumar Gyawali, Gianfranco Doretto, Donald A. Adjeroh
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: codecarbon, kmeans_gpu, matplotlib, numpy, optuna, pandas, psutil, scikit-learn, scipy, torch, tqdm
Required-by: 


# Smoke Test

In [5]:
import os
# Suppress Python warnings before importing third-party libraries.
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings
warnings.simplefilter("ignore")
import numpy as np
import torch
from bstabdiff import BSTabDiff

# ============================================================
# Synthetic smoke-test data
# ============================================================

rng = np.random.default_rng(0)
n, m = 20, 100
X = rng.standard_normal((n, m)).astype(np.float32)
y = rng.integers(0, 2, size=n)

# ============================================================
# Fit BSTabDiff
# ============================================================

model = BSTabDiff(
    M=10,
    prior_type="diffusion",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=0,
    prior_epochs=5,
    prior_batch=16,
    verbose_every=1,
    use_gobs=False,
)

returned = model.fit(X, y)

# ============================================================
# Basic sklearn-style checks
# ============================================================
assert returned is model
assert model.n_features_in_ == m
assert hasattr(model, "generator_")
assert hasattr(model, "train_info_")

# ============================================================
# Generate synthetic samples
# ============================================================

X_syn, R_syn, y_syn = model.sample(
    n_samples=5,
    return_mask=True,
)

# ============================================================
# Shape checks
# ============================================================

assert X_syn.shape == (5, m)
assert R_syn.shape == (5, m)
assert y_syn is not None
assert y_syn.shape == (5,)

print("\n" + "=" * 70)
print("Plain BSTabDiff smoke test OK")
print("=" * 70)
print("Device      :", model.device_)
print("Synthetic X:", X_syn.shape)
print("Synthetic R:", R_syn.shape)
print("Synthetic y:", y_syn.shape)

[prior:diffusion] epoch 1/5 | loss=1.219534
[prior:diffusion] epoch 2/5 | loss=1.039154
[prior:diffusion] epoch 3/5 | loss=1.212025
[prior:diffusion] epoch 4/5 | loss=1.136807
[prior:diffusion] epoch 5/5 | loss=1.146113

Plain BSTabDiff smoke test OK
Device      : cuda
Synthetic X: (5, 100)
Synthetic R: (5, 100)
Synthetic y: (5,)


**BSTabDiff w/ GO-BS Smoke Test**

In [6]:
import os
# Suppress Python warnings before importing third-party libraries.
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings
warnings.simplefilter("ignore")
import numpy as np
import torch
from bstabdiff import BSTabDiff

# ============================================================
# Synthetic smoke-test data
# ============================================================

rng = np.random.default_rng(0)
n, m = 20, 100
X = rng.standard_normal((n, m)).astype(np.float32)
y = rng.integers(0, 2, size=n)

# ============================================================
# Fit BSTabDiff with GO-BS
# ============================================================


model = BSTabDiff(
    M=10,
    prior_type="diffusion",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=0,
    prior_epochs=5,
    prior_batch=16,
    verbose_every=1,


    # GO-BS
    use_gobs=True,
    use_gobs_fc=False,
    gobs_num_clusters=3,
)

returned = model.fit(X, y)

# ============================================================
# Basic sklearn-style checks
# ============================================================

assert returned is model
assert model.n_features_in_ == m
assert hasattr(model, "generator_")
assert hasattr(model, "train_info_")

# ============================================================
# Generate synthetic samples
# ============================================================

X_syn, R_syn, y_syn = model.sample(
    n_samples=5,
    return_mask=True,
)

# ============================================================
# Shape checks
# ============================================================

assert X_syn.shape == (5, m)
assert R_syn.shape == (5, m)
assert y_syn is not None
assert y_syn.shape == (5,)

print("\n" + "=" * 70)
print("BSTabDiff + GO-BS smoke test OK")
print("=" * 70)
print("Device      :", model.device_)
print("Synthetic X:", X_syn.shape)
print("Synthetic R:", R_syn.shape)
print("Synthetic y:", y_syn.shape)

[prior:diffusion] epoch 1/5 | loss=1.223016
[prior:diffusion] epoch 2/5 | loss=1.038812
[prior:diffusion] epoch 3/5 | loss=1.211769
[prior:diffusion] epoch 4/5 | loss=1.138422
[prior:diffusion] epoch 5/5 | loss=1.149910

BSTabDiff + GO-BS smoke test OK
Device      : cuda
Synthetic X: (5, 100)
Synthetic R: (5, 100)
Synthetic y: (5,)


**BSTabDiff w/ GO-BS-FC Smoke Test**

In [7]:
import os
# Suppress Python warnings before importing third-party libraries.
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings
warnings.simplefilter("ignore")
import numpy as np
import torch
from bstabdiff import BSTabDiff

# ============================================================
# Synthetic smoke-test data
# ============================================================

rng = np.random.default_rng(0)
n, m = 20, 100
X = rng.standard_normal((n, m)).astype(np.float32)
y = rng.integers(0, 2, size=n)

# ============================================================
# Fit BSTabDiff with GO-BS-FC
# ============================================================

model = BSTabDiff(
    M=10,
    prior_type="diffusion",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=0,
    prior_epochs=5,
    prior_batch=16,
    verbose_every=1,

    # GO-BS-FC
    use_gobs=True,
    use_gobs_fc=True,
    gobs_num_clusters=3,
)

returned = model.fit(X, y)

# ============================================================
# Basic sklearn-style checks
# ============================================================

assert returned is model
assert model.n_features_in_ == m
assert hasattr(model, "generator_")
assert hasattr(model, "train_info_")

# ============================================================
# Generate synthetic samples
# ============================================================

X_syn, R_syn, y_syn = model.sample(
    n_samples=5,
    return_mask=True,
)

# ============================================================
# Shape checks
# ============================================================

assert X_syn.shape == (5, m)
assert R_syn.shape == (5, m)
assert y_syn is not None
assert y_syn.shape == (5,)

print("\n" + "=" * 70)
print("BSTabDiff + GO-BS-FC smoke test OK")
print("=" * 70)
print("Device      :", model.device_)
print("Synthetic X:", X_syn.shape)
print("Synthetic R:", R_syn.shape)
print("Synthetic y:", y_syn.shape)

[prior:diffusion] epoch 1/5 | loss=1.217432
[prior:diffusion] epoch 2/5 | loss=1.034602
[prior:diffusion] epoch 3/5 | loss=1.210683
[prior:diffusion] epoch 4/5 | loss=1.135645
[prior:diffusion] epoch 5/5 | loss=1.151312

BSTabDiff + GO-BS-FC smoke test OK
Device      : cuda
Synthetic X: (5, 100)
Synthetic R: (5, 100)
Synthetic y: (5,)


**w/ GO-BS and GO-BS-FC Smoke test**

In [8]:
import os, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")

import numpy as np
import torch
from sklearn.base import clone
from bstabdiff import BSTabDiff

SEED = 0
N = 20
M_FEATURES = 100
N_BLOCKS = 10
N_SYN = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

rng = np.random.default_rng(SEED)
X = rng.standard_normal((N, M_FEATURES)).astype(np.float32)
y = rng.integers(0, 2, size=N)

print("=" * 72)
print("BSTabDiff GO-BS / GO-BS-FC smoke tests")
print("=" * 72)
print("Device :", DEVICE)
print("X shape:", X.shape)
print("y shape:", y.shape)

def run_smoke_test(name, use_gobs, use_gobs_fc):
    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)

    model = BSTabDiff(
        M=N_BLOCKS,
        prior_type="diffusion",
        device=DEVICE,
        random_state=SEED,
        prior_epochs=5,
        prior_batch=16,
        prior_lr=1e-3,
        verbose_every=1,
        save_dir=None,
        save_best=True,
        use_ema=True,
        ema_decay=0.999,
        use_gobs=use_gobs,
        use_gobs_fc=use_gobs_fc,
        gobs_num_clusters=3,
        gobs_metric="kl_divergence",
        gobs_bins=32,
        gobs_top_k=None,
        gobs_refine_order=True,
        gobs_direction_select=True,
        gobs_refine_passes=1,
        gobs_boundary_refine_passes=5,
        gobs_boundary_window=8,
        gobs_lambda_cross=1.0,
        gobs_gamma_balance=1e-3,
        gobs_use_cpu_kmeans=False
    )

    returned = model.fit(X, y)

    assert returned is model
    assert hasattr(model, "generator_")
    assert hasattr(model, "train_info_")
    assert model.n_features_in_ == M_FEATURES

    X_syn, R_syn, y_syn = model.sample(
        n_samples=N_SYN,
        return_mask=True
    )

    assert X_syn.shape == (N_SYN, M_FEATURES)
    assert R_syn.shape == (N_SYN, M_FEATURES)
    assert y_syn is not None
    assert y_syn.shape == (N_SYN,)
    assert np.isfinite(X_syn[R_syn.astype(bool)]).all()

    params = model.get_params()
    assert params["M"] == N_BLOCKS
    assert params["use_gobs"] == use_gobs
    assert params["use_gobs_fc"] == use_gobs_fc

    cloned = clone(model)
    assert cloned.M == model.M
    assert cloned.use_gobs == model.use_gobs
    assert cloned.use_gobs_fc == model.use_gobs_fc

    print("Smoke test OK")
    print("Synthetic X :", X_syn.shape)
    print("Synthetic R :", R_syn.shape)
    print("Synthetic y :", y_syn.shape)
    print("Observed frac:", f"{R_syn.mean():.6f}")
    print("use_gobs     :", model.use_gobs)
    print("use_gobs_fc  :", model.use_gobs_fc)
    print("Device       :", model.device_)

    return model

model_gobs = run_smoke_test(
    name="BSTabDiff + GO-BS",
    use_gobs=True,
    use_gobs_fc=False
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_gobs_fc = run_smoke_test(
    name="BSTabDiff + GO-BS-FC",
    use_gobs=True,
    use_gobs_fc=True
)

print("\n" + "=" * 72)
print("ALL BSTabDiff ordering smoke tests PASSED")
print("=" * 72)
print("✓ GO-BS")
print("✓ GO-BS-FC")
print("✓ sklearn fit()")
print("✓ sample()")
print("✓ return_mask=True")
print("✓ get_params()")
print("✓ clone()")

BSTabDiff GO-BS / GO-BS-FC smoke tests
Device : cuda
X shape: (20, 100)
y shape: (20,)

BSTabDiff + GO-BS
[prior:diffusion] epoch 1/5 | loss=1.223016
[prior:diffusion] epoch 2/5 | loss=1.038812
[prior:diffusion] epoch 3/5 | loss=1.211769
[prior:diffusion] epoch 4/5 | loss=1.138422
[prior:diffusion] epoch 5/5 | loss=1.149910
Smoke test OK
Synthetic X : (5, 100)
Synthetic R : (5, 100)
Synthetic y : (5,)
Observed frac: 1.000000
use_gobs     : True
use_gobs_fc  : False
Device       : cuda

BSTabDiff + GO-BS-FC
[prior:diffusion] epoch 1/5 | loss=1.217432
[prior:diffusion] epoch 2/5 | loss=1.034602
[prior:diffusion] epoch 3/5 | loss=1.210683
[prior:diffusion] epoch 4/5 | loss=1.135645
[prior:diffusion] epoch 5/5 | loss=1.151312
Smoke test OK
Synthetic X : (5, 100)
Synthetic R : (5, 100)
Synthetic y : (5,)
Observed frac: 1.000000
use_gobs     : True
use_gobs_fc  : True
Device       : cuda

ALL BSTabDiff ordering smoke tests PASSED
✓ GO-BS
✓ GO-BS-FC
✓ sklearn fit()
✓ sample()
✓ return_mask=Tr

**Example 8 - Mixed Type Data with Missingness || Dummy Dataset**

In [9]:
import os, random, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch
from bstabdiff import BSTabDiff, FeatureSpec


SEED = 42
N = 80
N_CONT = 200
N_CAT = 10
N_SYN = 100
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_default_M(n, m):
    rho = m / n
    kappa = n * m
    if m <= 64:
        return m
    if rho < 10:
        return 64
    if kappa < 2.5e5:
        M = 32
    elif kappa < 1e6:
        M = 64
    elif kappa < 3e6:
        M = 128
    else:
        M = 192
    return min(M, m)


def make_dummy_mixed_data():
    rng = np.random.default_rng(SEED)
    y = rng.integers(0, 2, size=N)
    z = rng.normal(size=N)


    data = {}
    for j in range(N_CONT):
        scale = 0.3 + 0.7 * rng.random()
        signal = (0.2 + 0.8 * rng.random()) * y
        data[f"cont_{j}"] = scale * z + signal + rng.normal(0, 0.7, N)


    cat_levels = {
        "sex": ["Female", "Male"],
        "smoking": ["Never", "Former", "Current"],
        "stage": ["I", "II", "III", "IV"],
        "site": ["A", "B", "C"],
        "therapy": ["None", "DrugA", "DrugB"],
        "status": ["Negative", "Positive"],
        "group": ["G1", "G2", "G3", "G4"],
        "region": ["North", "South", "East", "West"],
        "marker": ["Low", "Medium", "High"],
        "batch": ["B1", "B2", "B3"]
    }


    for col, levels in cat_levels.items():
        K = len(levels)
        codes = (rng.integers(0, K, size=N) + y) % K
        data[col] = np.asarray(levels, dtype=object)[codes]


    df = pd.DataFrame(data)
    df["label"] = y


    cont_cols = [f"cont_{j}" for j in range(N_CONT)]
    cat_cols = list(cat_levels)


    for col in cont_cols:
        p = 0.08 + 0.07 * y
        miss = rng.random(N) < p
        df.loc[miss, col] = np.nan


    for col in cat_cols:
        p = 0.10 + 0.10 * y
        miss = rng.random(N) < p
        df.loc[miss, col] = np.nan


    return df, cont_cols, cat_cols


def encode_mixed_data(df, cont_cols, cat_cols):
    feature_names = cont_cols + cat_cols
    X = np.full((len(df), len(feature_names)), np.nan, dtype=np.float32)
    feature_specs = []
    category_maps = {}


    for j, col in enumerate(feature_names):
        if col in cat_cols:
            codes, levels = pd.factorize(df[col], sort=True)
            obs = codes >= 0
            X[obs, j] = codes[obs].astype(np.float32)
            category_maps[col] = list(levels)
            feature_specs.append(
                FeatureSpec(name=col, kind="categorical", n_categories=len(levels))
            )
        else:
            X[:, j] = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float32)
            feature_specs.append(FeatureSpec(name=col, kind="continuous"))


    y = df["label"].to_numpy(dtype=int)
    return X, y, feature_names, feature_specs, category_maps


def decode_synthetic(X_syn, y_syn, feature_names, category_maps):
    df_syn = pd.DataFrame(X_syn, columns=feature_names)
    for col, levels in category_maps.items():
        vals = df_syn[col].to_numpy(dtype=float)
        obs = np.isfinite(vals)
        decoded = np.full(len(vals), np.nan, dtype=object)
        codes = np.rint(vals[obs]).astype(int)
        decoded[obs] = [levels[c] for c in codes]
        df_syn[col] = decoded
    if y_syn is not None:
        df_syn["label"] = np.asarray(y_syn).astype(int)
    return df_syn


set_seed(SEED)


df_real, continuous_cols, categorical_cols = make_dummy_mixed_data()
X, y, feature_names, feature_specs, category_maps = encode_mixed_data(
    df_real, continuous_cols, categorical_cols
)


n, m = X.shape
M = select_default_M(n, m)


print(f"Dataset: n={n}, m={m}, rho={m/n:.2f}, kappa={n*m}, M={M}")
print(f"Continuous features: {len(continuous_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Classes: {np.unique(y).tolist()}")
print(f"Missing cells: {np.isnan(X).sum()} ({100*np.isnan(X).mean():.2f}%)")
print(f"Device: {DEVICE}")


print("\nCategorical mappings:")
for col, levels in category_maps.items():
    print(f"{col}: {levels}")


model = BSTabDiff(
    feature_specs=feature_specs,
    M=M,
    blocks=None,
    permute_features=False,
    prior_type="diffusion",
    device=DEVICE,
    random_state=SEED,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_dummy_mixed",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    use_gobs=False,
    use_gobs_fc=False
)


model.fit(X, y)
X_syn, R_syn, y_syn = model.sample(n_samples=N_SYN, return_mask=True)


X_syn = np.asarray(X_syn, dtype=np.float32)
R_syn = np.asarray(R_syn)
y_syn = np.asarray(y_syn).astype(int)


print("\nSynthetic output:")
print("X_syn:", X_syn.shape)
print("R_syn:", R_syn.shape)
print("y_syn:", y_syn.shape)
print(f"Synthetic missing cells: {np.isnan(X_syn).sum()} ({100*np.isnan(X_syn).mean():.2f}%)")


for j, fs in enumerate(feature_specs):
    if fs.kind == "categorical":
        vals = X_syn[:, j]
        vals = vals[np.isfinite(vals)]
        assert np.all(vals == np.floor(vals))
        assert np.all((vals >= 0) & (vals < fs.n_categories))


df_syn = decode_synthetic(X_syn, y_syn, feature_names, category_maps)


print("\nReal mixed data:")
print(df_real[continuous_cols[:3] + categorical_cols[:4] + ["label"]].head())


print("\nSynthetic mixed data:")
print(df_syn[continuous_cols[:3] + categorical_cols[:4] + ["label"]].head())


df_real.to_csv("bstabdiff_dummy_mixed_real.csv", index=False)
df_syn.to_csv("bstabdiff_dummy_mixed_synthetic.csv", index=False)
pd.DataFrame(R_syn, columns=feature_names).to_csv(
    "bstabdiff_dummy_mixed_mask.csv", index=False
)


print("\nSaved:")
print("bstabdiff_dummy_mixed_real.csv")
print("bstabdiff_dummy_mixed_synthetic.csv")
print("bstabdiff_dummy_mixed_mask.csv")

Dataset: n=80, m=210, rho=2.62, kappa=16800, M=64
Continuous features: 200
Categorical features: 10
Classes: [0, 1]
Missing cells: 1977 (11.77%)
Device: cuda:0

Categorical mappings:
sex: ['Female', 'Male']
smoking: ['Current', 'Former', 'Never']
stage: ['I', 'II', 'III', 'IV']
site: ['A', 'B', 'C']
therapy: ['DrugA', 'DrugB', 'None']
status: ['Negative', 'Positive']
group: ['G1', 'G2', 'G3', 'G4']
region: ['East', 'North', 'South', 'West']
marker: ['High', 'Low', 'Medium']
batch: ['B1', 'B2', 'B3']
[prior:diffusion] epoch 1/300 | loss=0.991440
[prior:diffusion] epoch 100/300 | loss=0.580394
[prior:diffusion] epoch 200/300 | loss=0.456283
[prior:diffusion] epoch 300/300 | loss=0.470165

Synthetic output:
X_syn: (100, 210)
R_syn: (100, 210)
y_syn: (100,)
Synthetic missing cells: 2554 (12.16%)

Real mixed data:
     cont_0    cont_1    cont_2     sex smoking stage site  label
0  0.420908       NaN  0.917227  Female   Never     I    A      0
1  1.679130  0.921090  1.413734    Male  Former

In [10]:

from bstabdiff import BlockSubunitGenerator, FeatureSpec, fit_block_subunit_generator
print("bstabdiff imported successfully")

bstabdiff imported successfully


**Previous ICLR Workshop version smoke test**

In [11]:
import numpy as np
from bstabdiff import FeatureSpec, fit_block_subunit_generator

# Dummy HDLSS data
np.random.seed(42)
n, m = 80, 2000
X = np.random.randn(n, m).astype(np.float32)
y = np.random.randint(0, 2, size=n)
X[np.random.rand(n, m) < 0.1] = np.nan

# Feature schema
feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(m)]

# Fit BSTabDiff
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=20,
    blocks=None,
    permute_features=False,
    prior_type="diffusion",
    device="cpu",
    seed=42,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_demo",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    return_train_info=True,
)

# Sample synthetic data
X_syn, R_syn, y_syn = gen.sample(n=50)

print("X_syn shape:", X_syn.shape)
print("R_syn shape:", R_syn.shape)
print("y_syn shape:", y_syn.shape if y_syn is not None else None)
print("Best training info:", train_info)

[prior:diffusion] epoch 1/300 | loss=1.089956
[prior:diffusion] epoch 100/300 | loss=0.555257
[prior:diffusion] epoch 200/300 | loss=0.570127
[prior:diffusion] epoch 300/300 | loss=0.566063
X_syn shape: (50, 2000)
R_syn shape: (50, 2000)
y_syn shape: (50,)
Best training info: {'best_epoch': 277, 'best_loss': 0.48506245017051697, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999, 'gobs_used': False, 'gobs_variant': None, 'use_gobs_fc': False}


In [12]:
import numpy as np
from bstabdiff import FeatureSpec, fit_block_subunit_generator

# -------------------------
# 1) Create a dummy HDLSS dataset
# -------------------------
np.random.seed(42)

n = 80    # number of samples
m = 20000   # number of features (HDLSS: n << m)

X = np.random.randn(n, m).astype(np.float32)

# binary labels
y = np.random.randint(0, 2, size=n)

# add some missingness
missing_mask = np.random.rand(n, m) < 0.1
X[missing_mask] = np.nan

# -------------------------
# 2) Define feature schema
# -------------------------
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# -------------------------
# 3) Fit BSTabDiff
# -------------------------
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=20,                       # latent block dimension
    prior_type="diffusion",     # or "flow"
    device="cpu",               # change to "cuda:0" if available
    seed=42,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_demo",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    return_train_info=True,
)

print("Training info:")
print(train_info)

# -------------------------
# 4) Sample synthetic data
# -------------------------
X_syn, R_syn, y_syn = gen.sample(n=50)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)
print("R_syn shape:", R_syn.shape)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

[prior:diffusion] epoch 1/300 | loss=1.087770
[prior:diffusion] epoch 100/300 | loss=0.609957
[prior:diffusion] epoch 200/300 | loss=0.565672
[prior:diffusion] epoch 300/300 | loss=0.567240
Training info:
{'best_epoch': 277, 'best_loss': 0.4772273004055023, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999, 'gobs_used': False, 'gobs_variant': None, 'use_gobs_fc': False}

Synthetic outputs:
X_syn shape: (50, 20000)
R_syn shape: (50, 20000)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.899364

First synthetic sample (first 10 features):
[-1.391971   -0.8645405   0.10322108  0.20136768  1.1143863  -1.6575179
         nan  0.95579743  0.8499241   0.7713028 ]


In [13]:
import numpy as np
from bstabdiff import (
    FeatureSpec,
    fit_block_subunit_generator,
)

# =========================================================
# 1) Create a dummy HDLSS dataset
# =========================================================
np.random.seed(42)

n = 80                    # number of samples
m = 2000                   # number of features (HDLSS: n << m)
n_classes = 2             # binary classification example
missing_rate = 0.10       # fraction of missing entries

# Real data matrix
X = np.random.randn(n, m).astype(np.float32)

# Labels
y = np.random.randint(0, n_classes, size=n)

# Add missing values
missing_mask = np.random.rand(n, m) < missing_rate
X[missing_mask] = np.nan

# =========================================================
# 2) Define feature schema
# =========================================================
# Here all features are continuous.
# For categorical features, use:
# FeatureSpec(name="f0", kind="categorical", n_categories=3)
feature_specs = [
    FeatureSpec(name=f"f{j}", kind="continuous")
    for j in range(m)
]

# =========================================================
# 3) BSTabDiff fitting configuration
# =========================================================
M = 20                         # number of latent blocks
blocks = None                  # None -> uses equal block partition automatically
permute_features = False       # whether to apply a random feature permutation
prior_type = "diffusion"       # choices: "diffusion" or "flow"
device = "cpu"                 # use "cuda:0" if GPU is available
seed = 42

# Prior training hyperparameters
prior_epochs = 300
prior_batch = 64
prior_lr = 1e-3
verbose_every = 100

# Checkpoint / best-model options
save_dir = None                # e.g. "checkpoints" to save best checkpoint
save_name = "bstabdiff_demo"
save_best = True

# EMA options (mainly useful for diffusion prior)
use_ema = True
ema_decay = 0.999

# Whether to return training diagnostics
return_train_info = True

# =========================================================
# 4) Fit BSTabDiff
# =========================================================
gen, train_info = fit_block_subunit_generator(
    X=X,
    feature_specs=feature_specs,
    y=y,
    M=M,
    blocks=blocks,
    permute_features=permute_features,
    prior_type=prior_type,
    device=device,
    seed=seed,
    prior_epochs=prior_epochs,
    prior_batch=prior_batch,
    prior_lr=prior_lr,
    verbose_every=verbose_every,
    save_dir=save_dir,
    save_name=save_name,
    save_best=save_best,
    use_ema=use_ema,
    ema_decay=ema_decay,
    return_train_info=return_train_info,
)

print("Training info:")
print(train_info)

# =========================================================
# 5) Sample synthetic data
# =========================================================
n_syn = 50

# If y is omitted, class labels are sampled automatically (when class-conditional)
X_syn, R_syn, y_syn = gen.sample(n=n_syn)

print("\nSynthetic outputs:")
print("X_syn shape:", X_syn.shape)   # synthetic data
print("R_syn shape:", R_syn.shape)   # observed/missing mask (1=observed, 0=missing)
print("y_syn shape:", None if y_syn is None else y_syn.shape)

print("\nFirst 5 synthetic labels:")
print(y_syn[:5] if y_syn is not None else None)

print("\nObserved fraction in synthetic mask:")
print(R_syn.mean())

print("\nFirst synthetic sample (first 10 features):")
print(X_syn[0, :10])

# =========================================================
# 6) Optional: sample from a fixed class
# =========================================================
X_syn_c1, R_syn_c1, y_syn_c1 = gen.sample(n=10, y=1)

print("\nFixed-class synthetic labels:")
print(y_syn_c1)

# =========================================================
# 7) Optional: inspect learned model structure
# =========================================================
print("\nModel summary:")
print("Number of features (m):", gen.m)
print("Number of blocks (M):", gen.M)
print("Prior type:", gen.prior_type)
print("Has permutation:", gen.perm is not None)

[prior:diffusion] epoch 1/300 | loss=1.089956
[prior:diffusion] epoch 100/300 | loss=0.555257
[prior:diffusion] epoch 200/300 | loss=0.570127
[prior:diffusion] epoch 300/300 | loss=0.566063
Training info:
{'best_epoch': 277, 'best_loss': 0.48506245017051697, 'best_ckpt_path': None, 'loaded_at_end': True, 'prior_type': 'diffusion', 'used_ema': True, 'ema_decay': 0.999, 'gobs_used': False, 'gobs_variant': None, 'use_gobs_fc': False}

Synthetic outputs:
X_syn shape: (50, 2000)
R_syn shape: (50, 2000)
y_syn shape: (50,)

First 5 synthetic labels:
[0 0 1 1 1]

Observed fraction in synthetic mask:
0.89968

First synthetic sample (first 10 features):
[-0.02387653 -0.3264404  -0.8905753  -1.3320502  -1.8623976   0.3787238
         nan -1.1465381  -1.6162746  -0.35329103]

Fixed-class synthetic labels:
[1 1 1 1 1 1 1 1 1 1]

Model summary:
Number of features (m): 2000
Number of blocks (M): 20
Prior type: diffusion
Has permutation: False


**The new NeurIPS version release contains scikit-learn style estimator which the previous release didn't have**

In [14]:
from importlib.metadata import version

print("BSTabDiff version:", version("bstabdiff"))

BSTabDiff version: 0.2.1
